# Lab 7 - Large Language Models (LLMs)

In this lab I try the three transformer families with small Hugging Face models: a decoder-only model (DistilGPT-2), an encoder-decoder model (FLAN-T5-small) and an encoder-only model (DistilBERT). Then I do the tasks at the end.

Libraries used: transformers, sentencepiece, torch.

## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder-decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |

## 1. Setup

Install the libraries first if they are missing: `pip install transformers sentencepiece torch`

In [1]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import warnings
warnings.filterwarnings("ignore")

from transformers import pipeline, set_seed
from transformers.utils import logging
from huggingface_hub.utils import disable_progress_bars

# only to keep the output clean (no download bars)
logging.set_verbosity_error()
logging.disable_progress_bar()
disable_progress_bars()
from huggingface_hub.utils import logging as hf_logging
hf_logging.set_verbosity_error()

# GPT-2 likes to generate many new lines, so I print the text on one line
def clean(text):
    return " ".join(text.split())

print("Libraries loaded successfully!")

Libraries loaded successfully!


# Part A - Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on the previous tokens. I use **DistilGPT-2**, a small GPT-style model. With `do_sample=False` it always takes the most likely next token, so the output is the same every time.

In [2]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(clean(output[0]["generated_text"]))

Natural language processing is a common feature of the language processing process.


# Part B - Encoder-Decoder Model

## Encoder-decoder example: Instruction / Text-to-Text Generation

The encoder reads the input and the decoder produces the output. I use **FLAN-T5-small**.

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Je s'agissant de la technologie de linguage naturelle.


# Part C - Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and build a representation of it, so they are good when the task is to understand or classify text. Here I use a DistilBERT sentiment model.

In [4]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

## Task 1 - Model Families

For each task I choose the most suitable model family.

In [5]:
answers = {
    "1. Sentiment classification": "Encoder-only (BERT / DistilBERT)",
    "2. Machine translation": "Encoder-decoder (T5 / BART)",
    "3. Open-ended text generation": "Decoder-only (GPT)",
    "4. Named Entity Recognition": "Encoder-only (BERT / RoBERTa)",
    "5. Text summarization": "Encoder-decoder (T5 / BART)",
}

for task, family in answers.items():
    print(f"{task:32s} -> {family}")

1. Sentiment classification      -> Encoder-only (BERT / DistilBERT)
2. Machine translation           -> Encoder-decoder (T5 / BART)
3. Open-ended text generation    -> Decoder-only (GPT)
4. Named Entity Recognition      -> Encoder-only (BERT / RoBERTa)
5. Text summarization            -> Encoder-decoder (T5 / BART)


Why I chose these:

- Sentiment classification and NER need the model to understand the whole input and give a label for the text or for each word, so an encoder-only model fits.
- Translation and summarization turn one text into a different text, so the encoder reads the input and the decoder writes the output.
- Open-ended generation just continues a prompt token by token, which is what a decoder-only model does.

## Task 2 - Encoder-only Model

Use the sentiment classifier on the three sentences and print the label and score for each.

In [6]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

for s in sentences:
    res = classifier(s)[0]
    print(f"{s}")
    print(f"   label: {res['label']}, score: {res['score']:.4f}")

The movie was excellent.
   label: POSITIVE, score: 0.9999
The lecture was difficult to understand.
   label: NEGATIVE, score: 0.9997
I enjoyed learning about transformers.
   label: POSITIVE, score: 0.9997


## Task 3 - Encoder-Decoder Model

Use FLAN-T5 to summarize a short paragraph.

In [7]:
paragraph = (
    "Natural language processing is a field of artificial intelligence that helps computers "
    "understand human language. It is used in many applications such as translation, chatbots, "
    "search engines and spam filters. In the last years transformer models made these systems "
    "much more accurate."
)

prompt = "Summarize: " + paragraph

inputs = tokenizer(prompt, return_tensors="pt")
summary_ids = model.generate(**inputs, max_new_tokens=60)

print("Summary:")
print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))

Summary:
Learn about natural language processing.


## Task 5 - Short Reflection

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**

A chatbot has to write its answer one token at a time, using the conversation so far as the context. A decoder-only model is trained for exactly that: predict the next token from the previous ones. Because of this the same model can take the question and the earlier messages as one long prompt and just keep generating the reply. There is no separate encoder to run, so the design is simple and it scales well to very large models trained on a lot of text. That is why GPT-style models are used for conversation.

# Optional Challenge

I generate text 3 times with the same prompt and **without** `set_seed(42)`.

In [8]:
prompt = "Natural language processing is"

for run in range(1, 4):
    out = generator(prompt, max_new_tokens=30, do_sample=True, top_k=50)
    print(f"Run {run}: {clean(out[0]['generated_text'])}")
    print()

Run 1: Natural language processing is a new phenomenon in the digital age.

Run 2: Natural language processing is often performed with a few hundred of the above-mentioned keywords. This may be done by using a combination of various languages, such as Chinese, Japanese

Run 3: Natural language processing is a powerful and effective way to understand a language. It can help you understand a language, to understand the syntax, to understand the language, to understand



### Question

Why do you sometimes get different outputs even when the prompt is the same?

With sampling (`do_sample=True`) the model does not always take the most likely token. It gets a probability for every possible next token and picks one randomly according to those probabilities. Without a fixed seed the random choice is different in every run, so the text goes in a different direction each time. With `do_sample=False` (greedy) or with `set_seed(42)` the output is the same every time.

## Observations

- DistilGPT-2 with `do_sample=False` gave a short sentence that repeats ideas ("language processing process"). Greedy decoding is repeatable but it is a bit boring.
- FLAN-T5-small translated the sentence to French, but the French is not very good ("Je s'agissant de..."), because the model is very small.
- DistilBERT classified the three sentences correctly (positive, negative, positive) with scores above 0.99.
- The FLAN-T5 summary is only "Learn about natural language processing." It is short and misses the applications, which is expected from the small model.
- The 3 runs without a seed are all different from each other, which answers the optional question: sampling picks the next token randomly. The model is also small, so some outputs do not make much sense.
